Udacity Pytorch project 2: CIFAR-10 Image Classification

Some of the benchmark results on CIFAR-10 include:

 

78.9% Accuracy | [Deep Belief Networks; Krizhevsky, 2010](https://www.cs.toronto.edu/~kriz/conv-cifar10-aug2010.pdf)

 

90.6% Accuracy | [Maxout Networks; Goodfellow et al., 2013](https://arxiv.org/pdf/1302.4389.pdf)

 

96.0% Accuracy | [Wide Residual Networks; Zagoruyko et al., 2016](https://arxiv.org/pdf/1605.07146.pdf)

 

99.0% Accuracy | [GPipe; Huang et al., 2018](https://arxiv.org/pdf/1811.06965.pdf)

 

98.5% Accuracy | [Rethinking Recurrent Neural Networks and other Improvements for ImageClassification; Nguyen et al., 2020](https://arxiv.org/pdf/2007.15161.pdf)

In [9]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transform
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets
from torch.utils.data import DataLoader
import sys 
from torchinfo import summary


In [10]:
BATCH_SIZE = 4
DEVICE = 'cpu'

Load the Dataset

Specific your transforms as a list first. The transforms module is already loaded as transforms
CIFAR-10 is fortunately included in the torchvision module. Then you can create your dataset using the CIFAR10 object from torchvision.datasets. Make sure to specify download=True. 
Once your dataset is created, you'll also need to define a Dataloader from the torch.utils.data module for both train and test set



In [11]:
#Define transforms

# TODO: Define transforms for the training data and testing data
#these transforms are tentative!! Putting something in to get the ball rolling. They may very well suck. 
train_transforms = transform.Compose([ transform.RandomHorizontalFlip(),
                                       transform.ToTensor()]) 

test_transforms =  transform.Compose([   transform.ToTensor()])





#Create training set and define training data loader
#changing batch size from 64 to 32. not sure if I'll need to use a GPU or not. 
trainset = datasets.CIFAR10('data/', download=True, train=True, transform=train_transforms)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)



#Create your test set and define test data loader
testset = datasets.CIFAR10('data/', download=True, train=False, transform=test_transforms)
testloader = torch.utils.data.DataLoader(testset, batch_size=BATCH_SIZE, shuffle=True)

trainsetview = datasets.CIFAR10('data/', download=True, train=True)
trainloaderview = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)



#the 10 classes in the dataset
classes=('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')


Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified


In [12]:
class_names = trainset.classes
class_names

['airplane',
 'automobile',
 'bird',
 'cat',
 'deer',
 'dog',
 'frog',
 'horse',
 'ship',
 'truck']

Explore the Dataset

Using matplotlib, numpy and torch, explore the dimensions of your data 

you can view images using the show5 function below - it takes a dataloader as an argument. Remember that normalized images will look really weird to you. You may want to try changing your transforms to view images.Typically using no transforms other than toTensor() works well for viewing - but not as well for training your network. If show5 doesn't work, go back and check your code for creating your dataloaders and your training/test sets.


In [13]:
print(len(trainset.data))
print(len(trainset.targets))
print(len(testset.data))
print(len(testset.targets))


50000
50000
10000
10000


In [14]:
#TODO I need to add dataloaders for train and test.  

In [15]:

# train_dl = DataLoader(trainset, 
#     batch_size=BATCH_SIZE,  
#     shuffle=True )
# print(len(train_dl))
# test_dl = DataLoader(testset,
#     batch_size=BATCH_SIZE,
#     shuffle=False 
# )
# print(len(test_dl))
# print("check for len match:",len(train_dl)*BATCH_SIZE)
# print("check for len match:",len(test_dl)*BATCH_SIZE)
# #note: last set less than 64 count, hence multiplying the above overcounts one batch.
# print(len(trainset)/len(train_dl))
# print((len(trainset)/len(train_dl)-63)*BATCH_SIZE)


In [16]:
def show5(img_loader):
    dataiter=iter(img_loader)

    batch=next(dataiter)
    labels=batch[1][0:5]
    image=batch[0][0:5]
    for i in range(5):
        print(classes[labels[i]])
        image=images[i].numpy()
        plt.imshow(image.T)
        plt.show()
    

Build your Neural Network

Using the layers in torch.nn (which has been imported as nn) and the torch.nn.Functional module (imported as F)
construct a neural network based on the parameters of the dataset. Feel free to consult a model of any architecture-feedforward, convolutional or even something more advanced.

In [18]:
#do more exploring... need to get proper shape before building the network. 
dataiter = iter(trainloader)
images, labels = next(dataiter)
print(type(images))
print(images.shape)
print(labels.shape)
print(len(images))
print(len(dataiter))





<class 'torch.Tensor'>
torch.Size([4, 3, 32, 32])
torch.Size([4])
4
12500


In [19]:
32*32

1024

In [20]:
#REFERENCE https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
#torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, groups=1, bias=True, padding_mode='zeros', device=None, dtype=None)

In [39]:
class SNN(nn.Module):
    """
    A simple convolutional neural network for image classification.
    Input: 1x28x28 grayscale images
    Output: 10 classes (digits 0-9)
    """

    def __init__(self):
        super(SNN, self).__init__()

        # First convolutional block
        # Input: 1 channel, Output: 16 channels, Kernel: 3x3
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # Reduces 28x28 to 14x14

        # Second convolutional block
        # Input: 16 channels, Output: 32 channels, Kernel: 3x3
        self.conv2 = nn.Conv2d(
            in_channels=16, out_channels=32, kernel_size=3, padding=1
        )
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)  # Reduces 14x14 to 7x7

        # Fully connected layer
        # Input: 32 * 7 * 7 = 1568 features, Output: 10 classes
        self.fc = nn.Linear(in_features=2*32*32, out_features=10)

    def forward(self, x):
        # Apply first conv block
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        # Apply second conv block
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        # Flatten for fully connected layer
        x = x.view(x.size(0), -1)

        # Apply fully connected layer
        x = self.fc(x)
        return x

Specify a loss function and an optimizer and instantiate the model.
if you use a less common loss function, please note why you chose that function in a comment. 

Running your Neural Network 

use whatever method you like to train your neural network and ensure you record the average loss at each epoch. Don't forget to use torch.device() and the .to() method for both your model and data if you are using GPU

if you want to print your loss during each epoch, you can use the enumerate function and print the loss after a set of batches. 250 batches works well for most people. 

Plot the training loss (and validation loss/accuracy, if recorded)

In [40]:
32*7*7

1568

In [41]:
model = SNN()
model

SNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=2048, out_features=10, bias=True)
)

In [44]:
learning_rate =0.001
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [46]:
"""
Train the model for one epoch.

Args:
    model: The neural network
    train_loader: Training data loader
    criterion: Loss function
    optimizer: Optimizer
    device: CPU or GPU

Returns:
    Average loss for the epoch
"""
device = 'cpu'
model.train()
total_loss = 0
correct = 0
total = 0

for batch_idx, (data, target) in enumerate(trainloader):
    data, target = data.to(device), target.to(device)

    # Forward pass
    output = model(data)
    loss = criterion(output, target)

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    _, predicted = output.max(1)
    total += target.size(0)
    correct += predicted.eq(target).sum().item()

    if (batch_idx + 1) % 100 == 0:
        print(f"Batch {batch_idx + 1}/{len(trainloader)}, Loss: {loss.item():.4f}")

avg_loss = total_loss / len(trainloader)
accuracy = 100.0 * correct / total

Batch 100/12500, Loss: 1.8455
Batch 200/12500, Loss: 1.7137
Batch 300/12500, Loss: 1.7806
Batch 400/12500, Loss: 2.2059
Batch 500/12500, Loss: 2.7396
Batch 600/12500, Loss: 2.5187
Batch 700/12500, Loss: 1.6917
Batch 800/12500, Loss: 2.4418
Batch 900/12500, Loss: 2.0223
Batch 1000/12500, Loss: 1.3622
Batch 1100/12500, Loss: 0.9988
Batch 1200/12500, Loss: 1.7723
Batch 1300/12500, Loss: 1.4615
Batch 1400/12500, Loss: 2.0931
Batch 1500/12500, Loss: 1.8188
Batch 1600/12500, Loss: 1.7701
Batch 1700/12500, Loss: 2.4453
Batch 1800/12500, Loss: 1.9747
Batch 1900/12500, Loss: 2.3816
Batch 2000/12500, Loss: 1.4659
Batch 2100/12500, Loss: 1.7219
Batch 2200/12500, Loss: 2.0041
Batch 2300/12500, Loss: 1.3629
Batch 2400/12500, Loss: 0.8050
Batch 2500/12500, Loss: 1.4007
Batch 2600/12500, Loss: 1.7081
Batch 2700/12500, Loss: 1.8777
Batch 2800/12500, Loss: 1.4999
Batch 2900/12500, Loss: 1.9978
Batch 3000/12500, Loss: 1.4923
Batch 3100/12500, Loss: 0.9336
Batch 3200/12500, Loss: 1.2194
Batch 3300/12500,

In [47]:
accuracy

51.36

In [48]:
model.eval()
test_loss = 0
correct = 0
total = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        test_loss += criterion(output, target).item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()

    accuracy = 100.0 * correct / total

In [49]:
accuracy

59.89